# Module 19: Decorators & Closures

Decorators are one of Python's most powerful features — they let you modify or enhance functions without changing their source code.

## 19.1 Functions Are First-Class Objects

In Python, functions are objects. You can assign them to variables, pass them as arguments, and return them from other functions:

In [ ]:
def greet(name):
    return f"Hello, {name}!"

# Assign to variable
say_hi = greet
print(say_hi("Alice"))   # Hello, Alice!

# Pass as argument
def apply_twice(func, value):
    return func(func(value))

def double(x): return x * 2

print(apply_twice(double, 3))   # 12  (double(double(3)))

# Return from function
def make_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier   # return the inner function!

times3 = make_multiplier(3)
times5 = make_multiplier(5)
print(times3(10))   # 30
print(times5(10))   # 50

## 19.2 Closures

A **closure** is a function that remembers variables from its enclosing scope, even after that scope has finished:

In [ ]:
def make_counter(start=0):
    count = start   # this variable is "closed over"
    
    def increment(step=1):
        nonlocal count      # modify the enclosing variable
        count += step
        return count
    
    def reset():
        nonlocal count
        count = start
    
    def get():
        return count
    
    return increment, reset, get

inc, rst, get = make_counter(10)
print(get())     # 10
print(inc())     # 11
print(inc(5))    # 16
rst()
print(get())     # 10 (reset to start)

## 19.3 Decorator Basics

A **decorator** is a function that takes a function, wraps it with extra behavior, and returns the wrapped version.

```python
@decorator
def my_function():
    ...

# Equivalent to:
def my_function():
    ...
my_function = decorator(my_function)
```

In [ ]:
# A simple logging decorator
def log_calls(func):
    """Decorator: print when the function is called."""
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}() with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__}() returned {result!r}")
        return result
    return wrapper

@log_calls
def add(a, b):
    return a + b

@log_calls
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

add(3, 4)
greet("Alice")
greet("Bob", greeting="Hi")

## 19.4 `functools.wraps` — Preserving Metadata

Without `@wraps`, decorated functions lose their `__name__` and `__doc__`. Always use it:

In [ ]:
from functools import wraps
import time

def timer(func):
    """Decorator: measure execution time."""
    @wraps(func)   # preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Sum integers from 0 to n."""
    return sum(range(n + 1))

result = slow_sum(1_000_000)
print(f"Result: {result}")
print(f"Function name: {slow_sum.__name__}")   # slow_sum (preserved!)

## 19.5 Decorators With Arguments (Decorator Factories)

In [ ]:
from functools import wraps

def repeat(times):
    """Decorator factory: repeat the function `times` times."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say(message):
    print(message)

say("Hello!")   # prints Hello! three times

# Retry decorator with max attempts
def retry(max_attempts=3, exceptions=(Exception,)):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts+1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        raise
                    print(f"Attempt {attempt} failed: {e}. Retrying...")
        return wrapper
    return decorator

## 19.6 Stacking Decorators

Multiple decorators are applied bottom-up:

In [ ]:
from functools import wraps

def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

@bold
@italic   # applied first (bottom-up)
def format_text(text):
    return text

print(format_text("Hello"))   # <b><i>Hello</i></b>

## 19.7 `functools` — Useful Utilities

In [ ]:
from functools import lru_cache, partial, reduce

# lru_cache — memoize a function automatically
@lru_cache(maxsize=None)
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print(fib(40))   # fast even for large n
print(fib.cache_info())

# partial — create a function with some args pre-filled
from functools import partial

def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube   = partial(power, exp=3)

print(square(5))  # 25
print(cube(3))    # 27

# reduce — apply a function cumulatively
from functools import reduce
product = reduce(lambda a, b: a * b, [1,2,3,4,5])
print(product)   # 120

## ✅ Module 19 Summary

| Concept | Key point |
|---------|----------|
| First-class functions | Functions are objects |
| Closure | Inner function remembers outer scope |
| Decorator | Wraps a function with extra behavior |
| `@wraps` | Preserve function metadata |
| Decorator factory | Decorator that takes arguments |
| `@lru_cache` | Auto-memoization |
| `partial` | Pre-fill function arguments |

---
### 🏋️ Practice!
Open **`exercises/ex_19_decorators_and_closures.py`**